In [3]:
import pandas as pd

crop_df = pd.read_csv("../../data/raw/crop_production.csv")

print(crop_df.shape)
print(crop_df.columns.tolist())
crop_df.head()

(246091, 8)
['index', 'State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop', 'Area', 'Production']


,index,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production
0,0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0
1,1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0
2,2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0
3,3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0,641.0
4,4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0


In [4]:
print(crop_df.columns.tolist())

['index', 'State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop', 'Area', 'Production']


In [6]:
crop_df = crop_df.drop(columns=['index'])

In [7]:
print(crop_df.columns.tolist())

['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop', 'Area', 'Production']


In [8]:
district_registry = (
    crop_df[["State_Name", "District_Name"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["State_Name", "District_Name"])
    .reset_index(drop=True)
)

print("Districts:", district_registry.shape)
district_registry.head()

Districts: (652, 2)


,State_Name,District_Name
0,Andaman and Nicobar Islands,NICOBARS
1,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN
2,Andaman and Nicobar Islands,SOUTH ANDAMANS
3,Andhra Pradesh,ANANTAPUR
4,Andhra Pradesh,CHITTOOR


In [9]:
print(crop_df.shape)
print(crop_df.columns.tolist())
print(crop_df.head())

(246091, 7)
['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop', 'Area', 'Production']
                    State_Name District_Name  Crop_Year       Season  \
0  Andaman and Nicobar Islands      NICOBARS       2000  Kharif        
1  Andaman and Nicobar Islands      NICOBARS       2000  Kharif        
2  Andaman and Nicobar Islands      NICOBARS       2000  Kharif        
3  Andaman and Nicobar Islands      NICOBARS       2000  Whole Year    
4  Andaman and Nicobar Islands      NICOBARS       2000  Whole Year    

                  Crop    Area  Production  
0             Arecanut  1254.0      2000.0  
1  Other Kharif pulses     2.0         1.0  
2                 Rice   102.0       321.0  
3               Banana   176.0       641.0  
4            Cashewnut   720.0       165.0  


In [10]:
district_registry = (
    crop_df[
        ["State_Name", "District_Name"]
    ]
    .dropna()
    .drop_duplicates()
    .sort_values(
        ["State_Name", "District_Name"]
    )
    .reset_index(drop=True)
)

print("Shape:", district_registry.shape)

district_registry.head(20)

Shape: (652, 2)


,State_Name,District_Name
0,Andaman and Nicobar Islands,NICOBARS
1,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN
2,Andaman and Nicobar Islands,SOUTH ANDAMANS
3,Andhra Pradesh,ANANTAPUR
4,Andhra Pradesh,CHITTOOR
5,Andhra Pradesh,EAST GODAVARI
6,Andhra Pradesh,GUNTUR
7,Andhra Pradesh,KADAPA
8,Andhra Pradesh,KRISHNA
9,Andhra Pradesh,KURNOOL


In [11]:
district_registry.tail()

,State_Name,District_Name
647,West Bengal,MEDINIPUR EAST
648,West Bengal,MEDINIPUR WEST
649,West Bengal,MURSHIDABAD
650,West Bengal,NADIA
651,West Bengal,PURULIA


In [12]:
district_registry.to_csv(
    "../../data/processed/district_registry.csv",
    index=False
)

In [14]:
print("Unique states:", district_registry["State_Name"].nunique())
print("Unique districts:", district_registry["District_Name"].nunique())
print(district_registry.shape)

Unique states: 33
Unique districts: 646
(652, 2)


In [16]:
from geopy.geocoders import Nominatim
from time import sleep
import pandas as pd
import os

geolocator = Nominatim(
    user_agent="KrishiKalp-Crop-Recommendation"
)

results = []

for index, row in district_registry.iterrows():

    state = row["State_Name"]
    district = row["District_Name"]

    query = f"{district}, {state}, India"

    try:

        location = geolocator.geocode(
            query,
            timeout=10
        )

        if location:

            results.append({
                "State_Name": state,
                "District_Name": district,
                "latitude": location.latitude,
                "longitude": location.longitude
            })

            print(
                f"{index + 1}/{len(district_registry)} "
                f"✓ {district}, {state}"
            )

        else:

            results.append({
                "State_Name": state,
                "District_Name": district,
                "latitude": None,
                "longitude": None
            })

            print(
                f"{index + 1}/{len(district_registry)} "
                f"✗ {district}, {state}"
            )

    except Exception as e:

        results.append({
            "State_Name": state,
            "District_Name": district,
            "latitude": None,
            "longitude": None
        })

        print(
            f"{index + 1}/{len(district_registry)} "
            f"ERROR: {district}, {state}"
        )

    # Respect the geocoding service's rate limi

1/652 ✗ NICOBARS, Andaman and Nicobar Islands
2/652 ✓ NORTH AND MIDDLE ANDAMAN, Andaman and Nicobar Islands
3/652 ✗ SOUTH ANDAMANS, Andaman and Nicobar Islands
4/652 ✓ ANANTAPUR, Andhra Pradesh
5/652 ✓ CHITTOOR, Andhra Pradesh
6/652 ✓ EAST GODAVARI, Andhra Pradesh
7/652 ✓ GUNTUR, Andhra Pradesh
8/652 ✓ KADAPA, Andhra Pradesh
9/652 ✓ KRISHNA, Andhra Pradesh
10/652 ✓ KURNOOL, Andhra Pradesh
11/652 ✓ PRAKASAM, Andhra Pradesh
12/652 ✓ SPSR NELLORE, Andhra Pradesh
13/652 ✓ SRIKAKULAM, Andhra Pradesh
14/652 ✗ VISAKHAPATANAM, Andhra Pradesh
15/652 ✓ VIZIANAGARAM, Andhra Pradesh
16/652 ✓ WEST GODAVARI, Andhra Pradesh
17/652 ✓ ANJAW, Arunachal Pradesh
18/652 ✓ CHANGLANG, Arunachal Pradesh
19/652 ✓ DIBANG VALLEY, Arunachal Pradesh
20/652 ✓ EAST KAMENG, Arunachal Pradesh
21/652 ✓ EAST SIANG, Arunachal Pradesh
22/652 ✓ KURUNG KUMEY, Arunachal Pradesh
23/652 ✓ LOHIT, Arunachal Pradesh
24/652 ✓ LONGDING, Arunachal Pradesh
25/652 ✓ LOWER DIBANG VALLEY, Arunachal Pradesh
26/652 ✓ LOWER SUBANSIRI, Arun

In [17]:
district_registry_geocoded = pd.DataFrame(results)

print("Shape:", district_registry_geocoded.shape)

district_registry_geocoded.head()

Shape: (652, 4)


,State_Name,District_Name,latitude,longitude
0,Andaman and Nicobar Islands,NICOBARS,NaN,NaN
1,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN,12.611239,92.831654
2,Andaman and Nicobar Islands,SOUTH ANDAMANS,NaN,NaN
3,Andhra Pradesh,ANANTAPUR,14.678322,77.606504
4,Andhra Pradesh,CHITTOOR,13.325036,79.648061


In [18]:
print(
    district_registry_geocoded[
        district_registry_geocoded["latitude"].isna()
    ]
)

                      State_Name       District_Name  latitude  longitude
0    Andaman and Nicobar Islands            NICOBARS       NaN        NaN
2    Andaman and Nicobar Islands      SOUTH ANDAMANS       NaN        NaN
13                Andhra Pradesh      VISAKHAPATANAM       NaN        NaN
137                      Gujarat               DOHAD       NaN        NaN
183             Himachal Pradesh     LAHUL AND SPITI       NaN        NaN
201           Jammu and Kashmir           LEH LADAKH       NaN        NaN
216                    Jharkhand       EAST SINGHBUM       NaN        NaN
246                    Karnataka      DAKSHIN KANNAD       NaN        NaN
263                    Karnataka        UTTAR KANNAD       NaN        NaN
444                       Punjab           FIROZEPUR       NaN        NaN
536                   Telangana           RANGAREDDI       NaN        NaN
592                Uttar Pradesh         KUSHI NAGAR       NaN        NaN
610                Uttar Pradesh   SAN

In [19]:
print(
    "Missing coordinates:",
    district_registry_geocoded["latitude"].isna().sum()
)

Missing coordinates: 18


In [20]:
missing_districts = district_registry_geocoded[
    district_registry_geocoded["latitude"].isna()
]

print(missing_districts.to_string(index=False))

                 State_Name      District_Name  latitude  longitude
Andaman and Nicobar Islands           NICOBARS       NaN        NaN
Andaman and Nicobar Islands     SOUTH ANDAMANS       NaN        NaN
             Andhra Pradesh     VISAKHAPATANAM       NaN        NaN
                    Gujarat              DOHAD       NaN        NaN
           Himachal Pradesh    LAHUL AND SPITI       NaN        NaN
         Jammu and Kashmir          LEH LADAKH       NaN        NaN
                  Jharkhand      EAST SINGHBUM       NaN        NaN
                  Karnataka     DAKSHIN KANNAD       NaN        NaN
                  Karnataka       UTTAR KANNAD       NaN        NaN
                     Punjab          FIROZEPUR       NaN        NaN
                 Telangana          RANGAREDDI       NaN        NaN
              Uttar Pradesh        KUSHI NAGAR       NaN        NaN
              Uttar Pradesh  SANT KABEER NAGAR       NaN        NaN
                Uttarakhand       RUDRA PRAYAG  

In [21]:
missing_districts.to_csv(
    "../../data/processed/missing_district_coordinates.csv",
    index=False
)

In [22]:
print(missing_districts.shape)
print(missing_districts.to_string(index=False))

(18, 4)
                 State_Name      District_Name  latitude  longitude
Andaman and Nicobar Islands           NICOBARS       NaN        NaN
Andaman and Nicobar Islands     SOUTH ANDAMANS       NaN        NaN
             Andhra Pradesh     VISAKHAPATANAM       NaN        NaN
                    Gujarat              DOHAD       NaN        NaN
           Himachal Pradesh    LAHUL AND SPITI       NaN        NaN
         Jammu and Kashmir          LEH LADAKH       NaN        NaN
                  Jharkhand      EAST SINGHBUM       NaN        NaN
                  Karnataka     DAKSHIN KANNAD       NaN        NaN
                  Karnataka       UTTAR KANNAD       NaN        NaN
                     Punjab          FIROZEPUR       NaN        NaN
                 Telangana          RANGAREDDI       NaN        NaN
              Uttar Pradesh        KUSHI NAGAR       NaN        NaN
              Uttar Pradesh  SANT KABEER NAGAR       NaN        NaN
                Uttarakhand       RUDRA 

In [23]:
missing_coords = {
    ("Andaman and Nicobar Islands", "NICOBARS"):
        (9.15, 92.75),

    ("Andaman and Nicobar Islands", "SOUTH ANDAMANS"):
        (11.62, 92.73),

    ("Andhra Pradesh", "VISAKHAPATANAM"):
        (17.69, 83.22),

    ("Gujarat", "DOHAD"):
        (22.83, 74.26),

    ("Himachal Pradesh", "LAHUL AND SPITI"):
        (32.57, 77.03),

    ("Jammu and Kashmir", "LEH LADAKH"):
        (34.15, 77.58),

    ("Jharkhand", "EAST SINGHBUM"):
        (22.80, 86.20),

    ("Karnataka", "DAKSHIN KANNAD"):
        (12.91, 74.86),

    ("Karnataka", "UTTAR KANNAD"):
        (14.82, 74.13),

    ("Punjab", "FIROZEPUR"):
        (30.93, 74.61),

    ("Telangana", "RANGAREDDI"):
        (17.32, 78.40),

    ("Uttar Pradesh", "KUSHI NAGAR"):
        (26.75, 83.89),

    ("Uttar Pradesh", "SANT KABEER NAGAR"):
        (26.77, 83.71),

    ("Uttarakhand", "RUDRA PRAYAG"):
        (30.28, 78.98),

    ("Uttarakhand", "UDAM SINGH NAGAR"):
        (28.98, 79.40),

    ("Uttarakhand", "UTTAR KASHI"):
        (30.73, 78.44),

    ("West Bengal", "24 PARAGANAS NORTH"):
        (22.72, 88.48),

    ("West Bengal", "24 PARAGANAS SOUTH"):
        (22.53, 88.34)
}

In [24]:
for (state, district), (lat, lon) in missing_coords.items():

    mask = (
        (district_registry_geocoded["State_Name"] == state) &
        (district_registry_geocoded["District_Name"] == district)
    )

    district_registry_geocoded.loc[mask, "latitude"] = lat
    district_registry_geocoded.loc[mask, "longitude"] = lon

In [25]:
print(
    "Missing coordinates:",
    district_registry_geocoded["latitude"].isna().sum()
)

Missing coordinates: 2


In [26]:
missing_districts = district_registry_geocoded[
    district_registry_geocoded["latitude"].isna()
]

print(missing_districts.to_string(index=False))

        State_Name District_Name  latitude  longitude
Jammu and Kashmir     LEH LADAKH       NaN        NaN
        Telangana     RANGAREDDI       NaN        NaN


In [27]:
missing_coords = {
    ("Jammu and Kashmir", "LEH LADAKH"):
        (34.165, 77.584),

    ("Telangana", "RANGAREDDI"):
        (17.32, 78.40)
}

for (state, district), (lat, lon) in missing_coords.items():

    mask = (
        (district_registry_geocoded["State_Name"] == state) &
        (district_registry_geocoded["District_Name"] == district)
    )

    district_registry_geocoded.loc[mask, "latitude"] = lat
    district_registry_geocoded.loc[mask, "longitude"] = lon

In [30]:
print(
    "Missing coordinates:",
    district_registry_geocoded["latitude"].isna().sum()
)

print(
    "Shape:",
    district_registry_geocoded.shape
)

Missing coordinates: 2
Shape: (652, 4)


In [29]:
from pathlib import Path

output_path = Path("../../data/processed")
output_path.mkdir(parents=True, exist_ok=True)

district_registry_geocoded.to_csv(
    output_path / "district_registry_geocoded.csv",
    index=False
)

print(
    "Saved to:",
    (output_path / "district_registry_geocoded.csv").resolve()
)

Saved to: D:\VS CODE SAVES\NTCC\KrishKalp\KrishiKalp-Smart-Crop-Recommendation-website-\data\processed\district_registry_geocoded.csv


In [31]:
print(
    district_registry_geocoded[
        district_registry_geocoded["latitude"].isna()
    ].to_string(index=False)
)

print("\nExact values:")

for _, row in district_registry_geocoded[
    district_registry_geocoded["latitude"].isna()
].iterrows():

    print(repr(row["State_Name"]), repr(row["District_Name"]))

        State_Name District_Name  latitude  longitude
Jammu and Kashmir     LEH LADAKH       NaN        NaN
        Telangana     RANGAREDDI       NaN        NaN

Exact values:
'Jammu and Kashmir ' 'LEH LADAKH'
'Telangana ' 'RANGAREDDI'


In [32]:
district_registry_geocoded["State_Name"] = (
    district_registry_geocoded["State_Name"]
    .str.strip()
)

district_registry_geocoded["District_Name"] = (
    district_registry_geocoded["District_Name"]
    .str.strip()
)

In [33]:
district_registry_geocoded.loc[
    district_registry_geocoded["District_Name"] == "LEH LADAKH",
    ["latitude", "longitude"]
] = [34.165, 77.584]

district_registry_geocoded.loc[
    district_registry_geocoded["District_Name"] == "RANGAREDDI",
    ["latitude", "longitude"]
] = [17.32, 78.40]

In [34]:
print(
    "Missing coordinates:",
    district_registry_geocoded["latitude"].isna().sum()
)

print(
    "Shape:",
    district_registry_geocoded.shape
)

Missing coordinates: 0
Shape: (652, 4)


In [35]:
from pathlib import Path

output_path = Path("../../data/processed")
output_path.mkdir(parents=True, exist_ok=True)

district_registry_geocoded.to_csv(
    output_path / "district_registry_geocoded.csv",
    index=False
)

print(
    "Saved to:",
    (output_path / "district_registry_geocoded.csv").resolve()
)

Saved to: D:\VS CODE SAVES\NTCC\KrishKalp\KrishiKalp-Smart-Crop-Recommendation-website-\data\processed\district_registry_geocoded.csv
